# LangGraph OTEL Instrumentation API - Prototype

This notebook demonstrates the **simplified API** for instrumenting LangGraph agents with OpenTelemetry (OTEL) tracing.

## Key Features
- **One-liner instrumentation**: `instrument_graph()` wraps any LangGraph with full OTEL tracing
- **One-liner optimization**: `optimize_langgraph()` runs optimization loops with telemetry capture
- **Dual semantic conventions**: Emits spans compatible with both Trace TGJ and Agent Lightning
- **Flexible LLM backend**: Supports OpenRouter API or StubLLM for testing

## Table of Contents
1. [Install Dependencies](#1-install-dependencies)
2. [Configuration](#2-configuration)
3. [Core Components](#3-core-components)
4. [LangGraph Definition](#4-langgraph-definition)
5. [API Functions](#5-api-functions)
6. [Demo: Single Execution](#6-demo-single-execution)
7. [Demo: Optimization Loop](#7-demo-optimization-loop)
8. [View Trace Output](#8-view-trace-output)

## 1. Install Dependencies

Run this cell to install all required packages.

In [1]:
# Install required packages
!pip install langgraph>=1.0.0 python-dotenv>=1.0.0 requests>=2.28.0 typing_extensions>=4.0.0

print("\n" + "="*50)
print("All dependencies installed successfully!")
print("="*50)


All dependencies installed successfully!


**API keys (no keys in parameters):**
- **Google Colab:** Use [Colab Secrets](https://colab.research.google.com/notebooks/secrets.ipynb): add `OPENROUTER_API_KEY` in the notebook secret manager, then read with `userdata.get("OPENROUTER_API_KEY")` or set `os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY", "")`.
- **Local / .env:** Use a `.env` file and `python-dotenv` (or set `os.environ` manually). Never pass API keys as function parameters.

**Persistent output (Google Colab):** When running on Colab, the next cell mounts Google Drive and creates a run folder. All trace outputs will be saved there; the run folder path is printed so you can find results after closing the notebook.

In [ ]:
# Auto-save results to Google Drive when on Colab (persistent); print run folder path
import os
from datetime import datetime

RUN_FOLDER = None
try:
    import google.colab
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    base = "/content/drive/MyDrive/NewTrace_runs"
    os.makedirs(base, exist_ok=True)
    RUN_FOLDER = os.path.join(base, f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
    os.makedirs(RUN_FOLDER, exist_ok=True)
    print(f"Run folder (persistent): {RUN_FOLDER}")
except Exception:
    RUN_FOLDER = os.path.abspath(os.path.join(os.getcwd(), "notebook_outputs"))
    os.makedirs(RUN_FOLDER, exist_ok=True)
    print(f"Run folder (local): {RUN_FOLDER}")

## 2. Configuration

API keys are retrieved **automatically** (no keys in code):

| Priority | Source | How to set |
|----------|--------|------------|
| 1 | **Colab Secrets** | Click the key icon in the left sidebar → add `OPENROUTER_API_KEY` |
| 2 | **Environment variable** | `export OPENROUTER_API_KEY=sk-or-v1-...` in your shell |
| 3 | **`.env` file** | Create a `.env` file with `OPENROUTER_API_KEY=sk-or-v1-...` |

Set `USE_STUB_LLM = True` below to run without any API calls (deterministic test mode).

In [ ]:
from __future__ import annotations
import os

# =============================================================================
# CONFIGURATION
# =============================================================================
# API keys are loaded automatically:
#   1. Google Colab  -> Colab Secrets (add OPENROUTER_API_KEY in the key icon)
#   2. Local / CI    -> .env file or shell environment variable
# NEVER paste a key directly into this cell.
# =============================================================================

# Option 1: Use StubLLM (no API calls needed - good for testing)
USE_STUB_LLM = False

# Model to use (free tier available)
OPENROUTER_MODEL = "meta-llama/llama-3.1-8b-instruct:free"

# ---------- key retrieval (Colab Secrets → env → .env file) ----------
OPENROUTER_API_KEY = ""

# Try Colab Secrets first
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY") or ""
    if OPENROUTER_API_KEY:
        print("[INFO] API key loaded from Colab Secrets.")
except (ImportError, ModuleNotFoundError):
    pass  # Not running on Colab

# Fall back to existing environment variable
if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
    if OPENROUTER_API_KEY:
        print("[INFO] API key loaded from environment variable.")

# Fall back to .env file
if not OPENROUTER_API_KEY:
    try:
        from dotenv import load_dotenv
        load_dotenv()  # loads from .env in cwd or parent dirs
        OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
        if OPENROUTER_API_KEY:
            print("[INFO] API key loaded from .env file.")
    except ImportError:
        pass

if not OPENROUTER_API_KEY:
    print("[WARN] No OPENROUTER_API_KEY found. Will fall back to StubLLM.")

# Publish to env so downstream code (OpenRouterLLM, etc.) can read them
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
os.environ["OPENROUTER_MODEL"] = OPENROUTER_MODEL
os.environ["USE_STUB_LLM"] = str(USE_STUB_LLM).lower()

print("\nConfiguration:")
print(f"  USE_STUB_LLM: {USE_STUB_LLM}")
print(f"  OPENROUTER_API_KEY: {'[SET]' if OPENROUTER_API_KEY else '[NOT SET]'}")
print(f"  OPENROUTER_MODEL: {OPENROUTER_MODEL}")
print(f"\nMode: {'STUB LLM (no API calls)' if USE_STUB_LLM or not OPENROUTER_API_KEY else 'REAL LLM (OpenRouter)'}")

Configuration:
  USE_STUB_LLM: False
  OPENROUTER_API_KEY: [SET]
  OPENROUTER_MODEL: meta-llama/llama-3.1-8b-instruct:free

Mode: REAL LLM (OpenRouter)


## 3. Core Components

Import and define the core tracing components:
- `TelemetrySession` - OTEL span management
- `TracingLLM` - LLM wrapper with dual semantic conventions
- `OpenRouterLLM` / `StubLLM` - LLM backends

In [3]:
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Set
from pathlib import Path
import json
import time
import requests

# LangGraph imports
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict

print("Core imports loaded successfully!")

Core imports loaded successfully!


In [4]:
# =============================================================================
# OPENROUTER LLM CLIENT
# =============================================================================

class OpenRouterLLM:
    """
    LLM client for OpenRouter API.
    Compatible with OpenAI-style interface: response.choices[0].message.content
    """
    
    def __init__(self, api_key: Optional[str] = None, model: Optional[str] = None):
        self.api_key = api_key or os.environ.get("OPENROUTER_API_KEY", "")
        self.model = model or os.environ.get("OPENROUTER_MODEL", "meta-llama/llama-3.1-8b-instruct:free")
        self.base_url = "https://openrouter.ai/api/v1"
        self.call_count = 0
        
        if not self.api_key:
            raise ValueError("OpenRouter API key not provided.")
    
    def __call__(self, messages: List[Dict[str, str]], **kwargs) -> Any:
        self.call_count += 1
        
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
        }
        
        payload = {
            "model": kwargs.get("model", self.model),
            "messages": messages,
            "temperature": kwargs.get("temperature", 0.7),
            "max_tokens": kwargs.get("max_tokens", 1024),
        }
        
        try:
            response = requests.post(
                f"{self.base_url}/chat/completions",
                headers=headers,
                json=payload,
                timeout=60,
            )
            response.raise_for_status()
            data = response.json()
            return self._make_response(data)
        except Exception as e:
            print(f"[ERROR] API call failed: {e}")
            return self._make_response({"choices": [{"message": {"content": json.dumps({"error": str(e)})}}]})
    
    def _make_response(self, data: Dict[str, Any]) -> Any:
        class Message:
            def __init__(self, content): self.content = content
        class Choice:
            def __init__(self, content): self.message = Message(content)
        class Response:
            def __init__(self, choices_data):
                self.choices = [Choice(c.get("message", {}).get("content", "")) for c in choices_data]
        return Response(data.get("choices", [{"message": {"content": ""}}]))

print("OpenRouterLLM class defined!")

OpenRouterLLM class defined!


In [5]:
# =============================================================================
# STUB LLM (Deterministic responses for testing)
# =============================================================================

class StubLLM:
    """Deterministic LLM stub for testing without API calls."""
    
    def __init__(self):
        self.call_count = 0
        self.model = "stub-llm"
    
    def __call__(self, messages: List[Dict[str, str]], **kwargs) -> Any:
        self.call_count += 1
        user_msg = messages[-1].get("content", "") if messages else ""
        
        # Pattern-based responses
        if "plan" in user_msg.lower():
            content = json.dumps({
                "1": {"action": "research", "goal": "gather information"},
                "2": {"action": "synthesize", "goal": "create answer"}
            })
        elif "evaluat" in user_msg.lower():
            base_score = 0.7 + (self.call_count % 3) * 0.05
            content = json.dumps({
                "answer_relevance": round(base_score, 2),
                "groundedness": round(base_score - 0.05, 2),
                "plan_quality": round(base_score + 0.05, 2),
                "reasons": f"Evaluation {self.call_count}: Good structure."
            })
        else:
            content = f"Response #{self.call_count}: Based on the context, here is a comprehensive answer."
        
        return self._make_response(content)
    
    def _make_response(self, content: str) -> Any:
        class Message:
            def __init__(self, c): self.content = c
        class Choice:
            def __init__(self, c): self.message = Message(c)
        class Response:
            def __init__(self, c): self.choices = [Choice(c)]
        return Response(content)


def get_llm(use_stub: bool = False) -> Any:
    """Get LLM client based on configuration."""
    if use_stub or os.environ.get("USE_STUB_LLM", "").lower() in ("true", "1"):
        return StubLLM()
    if not os.environ.get("OPENROUTER_API_KEY"):
        print("[INFO] No API key found. Using StubLLM.")
        return StubLLM()
    return OpenRouterLLM()

print("StubLLM class defined!")

StubLLM class defined!


In [6]:
# =============================================================================
# TELEMETRY SESSION (OTEL span management)
# =============================================================================

class TelemetrySession:
    """Manages OTEL tracing session with export capabilities."""
    
    def __init__(self, service_name: str = "trace-session"):
        self.service_name = service_name
        self._spans: List[Dict[str, Any]] = []
        self._span_counter = 0
        self._trace_id = f"trace_{int(time.time() * 1000)}"
    
    def start_span(self, name: str) -> "SpanContext":
        """Start a new span and return context for attributes."""
        self._span_counter += 1
        span = {
            "traceId": self._trace_id,
            "spanId": f"span_{self._span_counter:04d}",
            "name": name,
            "startTimeUnixNano": time.time_ns(),
            "endTimeUnixNano": 0,
            "attributes": {},
        }
        self._spans.append(span)
        return SpanContext(span)
    
    def flush_otlp(self, clear: bool = True) -> Dict[str, Any]:
        """Export collected spans to OTLP JSON format."""
        for span in self._spans:
            if span["endTimeUnixNano"] == 0:
                span["endTimeUnixNano"] = time.time_ns()
        
        otlp_spans = []
        for span in self._spans:
            attrs = [{"key": k, "value": {"stringValue": str(v)}} for k, v in span["attributes"].items()]
            otlp_spans.append({
                "traceId": span["traceId"],
                "spanId": span["spanId"],
                "name": span["name"],
                "startTimeUnixNano": span["startTimeUnixNano"],
                "endTimeUnixNano": span["endTimeUnixNano"],
                "attributes": attrs,
            })
        
        result = {
            "resourceSpans": [{
                "resource": {"attributes": []},
                "scopeSpans": [{
                    "scope": {"name": self.service_name},
                    "spans": otlp_spans,
                }]
            }]
        }
        
        if clear:
            self._spans.clear()
            self._span_counter = 0
            self._trace_id = f"trace_{int(time.time() * 1000)}"
        
        return result


class SpanContext:
    """Context manager for span attribute setting."""
    
    def __init__(self, span: Dict[str, Any]):
        self._span = span
    
    def set_attribute(self, key: str, value: Any) -> None:
        self._span["attributes"][key] = value
    
    def end(self) -> None:
        self._span["endTimeUnixNano"] = time.time_ns()
    
    def __enter__(self) -> "SpanContext":
        return self
    
    def __exit__(self, *args) -> None:
        self.end()

print("TelemetrySession class defined!")

TelemetrySession class defined!


In [7]:
# =============================================================================
# TRACING LLM (Wrapper with dual semantic conventions)
# =============================================================================

class TracingLLM:
    """
    LLM wrapper with OTEL tracing and dual semantic conventions.
    Emits spans compatible with both Trace TGJ and Agent Lightning.
    """
    
    def __init__(self, llm: Any, session: TelemetrySession, *, 
                 trainable_keys: Optional[Set[str]] = None,
                 provider_name: str = "openrouter",
                 emit_genai_child_span: bool = True):
        self.llm = llm
        self.session = session
        self.trainable_keys = trainable_keys or set()
        self.provider_name = provider_name
        self.emit_genai_child_span = emit_genai_child_span
    
    def _is_trainable(self, key: Optional[str]) -> bool:
        if key is None:
            return False
        return "" in self.trainable_keys or key in self.trainable_keys
    
    def node_call(self, *, span_name: str, template_name: Optional[str] = None,
                  template: Optional[str] = None, optimizable_key: Optional[str] = None,
                  messages: Optional[List[Dict[str, str]]] = None, **llm_kwargs) -> str:
        """Invoke LLM under an OTEL span with full tracing."""
        messages = messages or []
        
        user_prompt = ""
        for msg in reversed(messages):
            if msg.get("role") == "user":
                user_prompt = msg.get("content", "")
                break
        
        # Parent span (Trace-compatible)
        with self.session.start_span(span_name) as sp:
            if template_name and template is not None:
                sp.set_attribute(f"param.{template_name}", template[:200])
                sp.set_attribute(f"param.{template_name}.trainable", str(self._is_trainable(optimizable_key)))
            
            sp.set_attribute("gen_ai.model", getattr(self.llm, "model", "llm"))
            sp.set_attribute("inputs.gen_ai.prompt", user_prompt[:300])
            
            # Child span (Agent Lightning-compatible)
            if self.emit_genai_child_span:
                with self.session.start_span(f"{self.provider_name}.chat.completion") as llm_sp:
                    llm_sp.set_attribute("trace.temporal_ignore", "true")
                    llm_sp.set_attribute("gen_ai.operation.name", "chat")
                    llm_sp.set_attribute("gen_ai.provider.name", self.provider_name)
                    
                    response = self.llm(messages=messages, **llm_kwargs)
                    content = response.choices[0].message.content
                    
                    llm_sp.set_attribute("gen_ai.output.preview", content[:200])
            else:
                response = self.llm(messages=messages, **llm_kwargs)
                content = response.choices[0].message.content
        
        return content

print("TracingLLM class defined!")

TracingLLM class defined!


## 4. LangGraph Definition

Define the research agent LangGraph with 4 nodes:
- **Planner**: Creates execution plan
- **Researcher**: Gathers information
- **Synthesizer**: Creates final answer
- **Evaluator**: Assesses answer quality

In [8]:
# =============================================================================
# LANGGRAPH STATE DEFINITION
# =============================================================================

class AgentState(TypedDict):
    """State for the research agent LangGraph."""
    query: str
    plan: Dict[str, Any]
    research_results: List[str]
    answer: str
    evaluation: Dict[str, Any]
    planner_template: str
    synthesizer_template: str


# Global references (set by instrument_graph)
_TRACING_LLM: Optional[TracingLLM] = None
_TEMPLATES: Dict[str, str] = {}

# Default templates
DEFAULT_PLANNER_TEMPLATE = """You are a planning agent. Given a user query, create a simple plan.
Output a JSON object with numbered steps.
User query: {query}
Respond with ONLY the JSON object."""

DEFAULT_SYNTHESIZER_TEMPLATE = """You are a synthesis agent. Given a query and research, provide a comprehensive answer.
Query: {query}
Research: {context}
Provide a clear, factual answer."""

DEFAULT_EVALUATOR_TEMPLATE = """Evaluate the answer quality on 0-1 scale.
Query: {query}
Answer: {answer}
Output JSON: {{"answer_relevance": 0.8, "groundedness": 0.7, "plan_quality": 0.9, "reasons": "..."}}"""

print("State and templates defined!")

State and templates defined!


In [9]:
# =============================================================================
# LANGGRAPH NODE FUNCTIONS
# =============================================================================

def planner_node(state: AgentState) -> Dict[str, Any]:
    """Planner node - creates execution plan."""
    global _TRACING_LLM, _TEMPLATES
    template = state.get("planner_template") or _TEMPLATES.get("planner_prompt", DEFAULT_PLANNER_TEMPLATE)
    prompt = template.replace("{query}", state["query"])
    
    response = _TRACING_LLM.node_call(
        span_name="planner",
        template_name="planner_prompt",
        template=template,
        optimizable_key="planner",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )
    
    try:
        plan = json.loads(response)
    except:
        plan = {"1": {"action": "synthesize", "goal": "answer directly"}}
    
    return {"plan": plan}


def researcher_node(state: AgentState) -> Dict[str, Any]:
    """Researcher node - gathers information."""
    global _TRACING_LLM
    response = _TRACING_LLM.node_call(
        span_name="researcher",
        messages=[{"role": "user", "content": f"Provide key facts about: {state['query']}"}],
        temperature=0.5,
    )
    return {"research_results": [response]}


def synthesizer_node(state: AgentState) -> Dict[str, Any]:
    """Synthesizer node - creates final answer."""
    global _TRACING_LLM, _TEMPLATES
    template = state.get("synthesizer_template") or _TEMPLATES.get("synthesizer_prompt", DEFAULT_SYNTHESIZER_TEMPLATE)
    context = "\n".join(state.get("research_results", ["No results."]))
    prompt = template.replace("{query}", state["query"]).replace("{context}", context)
    
    response = _TRACING_LLM.node_call(
        span_name="synthesizer",
        template_name="synthesizer_prompt",
        template=template,
        optimizable_key="synthesizer",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5,
    )
    return {"answer": response}


def evaluator_node(state: AgentState) -> Dict[str, Any]:
    """Evaluator node - assesses answer quality."""
    global _TRACING_LLM
    prompt = DEFAULT_EVALUATOR_TEMPLATE.replace("{query}", state["query"]).replace("{answer}", state.get("answer", ""))
    
    response = _TRACING_LLM.node_call(
        span_name="evaluator",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    
    try:
        evaluation = json.loads(response)
    except:
        evaluation = {"answer_relevance": 0.5, "groundedness": 0.5, "plan_quality": 0.5, "reasons": "Parse error"}
    
    return {"evaluation": evaluation}


def build_research_graph() -> StateGraph:
    """Build a real LangGraph for research tasks."""
    graph = StateGraph(AgentState)
    
    graph.add_node("planner", planner_node)
    graph.add_node("researcher", researcher_node)
    graph.add_node("synthesizer", synthesizer_node)
    graph.add_node("evaluator", evaluator_node)
    
    graph.add_edge(START, "planner")
    graph.add_edge("planner", "researcher")
    graph.add_edge("researcher", "synthesizer")
    graph.add_edge("synthesizer", "evaluator")
    graph.add_edge("evaluator", END)
    
    return graph

print("Node functions and graph builder defined!")

Node functions and graph builder defined!


## 5. API Functions

Define the main API functions:
- `instrument_graph()` - One-liner to add OTEL instrumentation
- `optimize_langgraph()` - One-liner for optimization loop

In [10]:
# =============================================================================
# INSTRUMENTED GRAPH WRAPPER
# =============================================================================

@dataclass
class InstrumentedGraph:
    """Instrumented LangGraph wrapper with telemetry."""
    graph: Any
    session: TelemetrySession
    tracing_llm: TracingLLM
    templates: Dict[str, str] = field(default_factory=dict)
    
    def invoke(self, state: Dict[str, Any]) -> Dict[str, Any]:
        """Execute graph and capture telemetry."""
        query = state.get("query", state.get("user_query", ""))
        
        initial_state: AgentState = {
            "query": query,
            "plan": {},
            "research_results": [],
            "answer": "",
            "evaluation": {},
            "planner_template": self.templates.get("planner_prompt", ""),
            "synthesizer_template": self.templates.get("synthesizer_prompt", ""),
        }
        
        final_state = self.graph.invoke(initial_state)
        
        evaluation = final_state.get("evaluation", {})
        metrics = {
            "answer_relevance": float(evaluation.get("answer_relevance", 0.5)),
            "groundedness": float(evaluation.get("groundedness", 0.5)),
            "plan_quality": float(evaluation.get("plan_quality", 0.5)),
        }
        score = sum(metrics.values()) / len(metrics)
        
        # Record evaluation span
        with self.session.start_span("evaluation_metrics") as sp:
            sp.set_attribute("eval.score", str(score))
            for k, v in metrics.items():
                sp.set_attribute(f"eval.{k}", str(v))
            
            with self.session.start_span("agentlightning.annotation") as reward_sp:
                reward_sp.set_attribute("trace.temporal_ignore", "true")
                reward_sp.set_attribute("agentlightning.reward.0.name", "final_score")
                reward_sp.set_attribute("agentlightning.reward.0.value", str(score))
        
        return {
            "answer": final_state.get("answer", ""),
            "plan": final_state.get("plan", {}),
            "score": score,
            "metrics": metrics,
        }

print("InstrumentedGraph class defined!")

InstrumentedGraph class defined!


In [11]:
# =============================================================================
# INSTRUMENT_GRAPH() - Main API
# =============================================================================

def instrument_graph(
    graph: Optional[StateGraph] = None,
    *,
    service_name: str = "langgraph-agent",
    trainable_keys: Optional[Set[str]] = None,
    llm: Optional[Any] = None,
    initial_templates: Optional[Dict[str, str]] = None,
    use_stub_llm: bool = False,
) -> InstrumentedGraph:
    """
    Wrap a LangGraph with automatic OTEL instrumentation.
    
    Parameters
    ----------
    graph : StateGraph, optional
        The LangGraph to instrument. If None, builds default research graph.
    service_name : str
        OTEL service name for trace identification.
    trainable_keys : Set[str], optional
        Node names whose prompts are trainable.
    llm : Any, optional
        LLM client. Uses OpenRouterLLM or StubLLM based on config.
    initial_templates : Dict[str, str], optional
        Initial prompt templates.
    use_stub_llm : bool
        If True, force use of StubLLM.
    
    Returns
    -------
    InstrumentedGraph
        Wrapper with invoke() and telemetry session.
    """
    global _TRACING_LLM, _TEMPLATES
    
    if graph is None:
        graph = build_research_graph()
    
    compiled_graph = graph.compile() if hasattr(graph, 'compile') else graph
    session = TelemetrySession(service_name)
    
    if llm is None:
        llm = get_llm(use_stub=use_stub_llm)
    
    tracing_llm = TracingLLM(
        llm=llm,
        session=session,
        trainable_keys=trainable_keys or {"planner", "synthesizer"},
        provider_name="openrouter" if isinstance(llm, OpenRouterLLM) else "stub",
    )
    
    _TRACING_LLM = tracing_llm
    _TEMPLATES = initial_templates or {}
    
    return InstrumentedGraph(
        graph=compiled_graph,
        session=session,
        tracing_llm=tracing_llm,
        templates=initial_templates or {},
    )

print("instrument_graph() function defined!")

instrument_graph() function defined!


In [12]:
# =============================================================================
# OPTIMIZE_LANGGRAPH() - Optimization Loop API
# =============================================================================

@dataclass
class RunResult:
    """Result of a single graph execution."""
    answer: str
    score: float
    metrics: Dict[str, float]
    otlp: Dict[str, Any]


@dataclass
class OptimizationResult:
    """Result of optimization loop."""
    baseline_score: float
    best_score: float
    best_iteration: int
    score_history: List[float]
    all_runs: List[List[RunResult]]


def optimize_langgraph(
    graph: InstrumentedGraph,
    queries: List[str],
    *,
    iterations: int = 3,
) -> OptimizationResult:
    """Run optimization loop on instrumented graph."""
    score_history = []
    all_runs = []
    best_score = 0.0
    best_iteration = 0
    
    # Baseline
    print("  Running baseline...")
    baseline_runs = []
    for i, q in enumerate(queries):
        print(f"    Query {i+1}/{len(queries)}: {q[:40]}...")
        result = graph.invoke({"query": q})
        baseline_runs.append(RunResult(
            answer=result["answer"],
            score=result["score"],
            metrics=result["metrics"],
            otlp=graph.session.flush_otlp(),
        ))
        print(f"      Score: {result['score']:.3f}")
    
    baseline_score = sum(r.score for r in baseline_runs) / len(baseline_runs)
    score_history.append(baseline_score)
    all_runs.append(baseline_runs)
    best_score = baseline_score
    print(f"  Baseline average: {baseline_score:.3f}")
    
    # Iterations
    for iteration in range(1, iterations + 1):
        print(f"\n  Iteration {iteration}/{iterations}...")
        runs = []
        for i, q in enumerate(queries):
            print(f"    Query {i+1}/{len(queries)}: {q[:40]}...")
            result = graph.invoke({"query": q})
            runs.append(RunResult(
                answer=result["answer"],
                score=result["score"],
                metrics=result["metrics"],
                otlp=graph.session.flush_otlp(),
            ))
            print(f"      Score: {result['score']:.3f}")
        
        iter_score = sum(r.score for r in runs) / len(runs)
        score_history.append(iter_score)
        all_runs.append(runs)
        
        if iter_score > best_score:
            best_score = iter_score
            best_iteration = iteration
            print(f"  Iteration {iteration} average: {iter_score:.3f} * NEW BEST")
        else:
            print(f"  Iteration {iteration} average: {iter_score:.3f}")
    
    return OptimizationResult(
        baseline_score=baseline_score,
        best_score=best_score,
        best_iteration=best_iteration,
        score_history=score_history,
        all_runs=all_runs,
    )

print("optimize_langgraph() function defined!")

optimize_langgraph() function defined!


## 6. Demo: Single Execution

Demonstrate single graph execution with OTEL tracing.

In [13]:
# =============================================================================
# DEMO: SINGLE EXECUTION
# =============================================================================

print("="*60)
print("DEMO: Single Graph Execution")
print("="*60)

# Step 1: Instrument the graph (ONE function call!)
print("\n1. Instrument a LangGraph:")
print("-" * 40)

instrumented = instrument_graph(
    service_name="demo-notebook",
    trainable_keys={"planner", "synthesizer"},
)

print(f"  -> Created InstrumentedGraph")
print(f"  -> Session: {instrumented.session.service_name}")
print(f"  -> LLM type: {type(instrumented.tracing_llm.llm).__name__}")

DEMO: Single Graph Execution

1. Instrument a LangGraph:
----------------------------------------
  -> Created InstrumentedGraph
  -> Session: demo-notebook
  -> LLM type: OpenRouterLLM


In [14]:
# Step 2: Execute the graph
print("\n2. Execute the graph:")
print("-" * 40)

test_query = "What are the main causes of climate change?"
print(f"  Query: {test_query}")

result = instrumented.invoke({"query": test_query})

print(f"\n  Results:")
print(f"    Score: {result['score']:.3f}")
print(f"    Metrics: {result['metrics']}")
print(f"    Answer: {result['answer'][:200]}...")


2. Execute the graph:
----------------------------------------
  Query: What are the main causes of climate change?
[ERROR] API call failed: 404 Client Error: Not Found for url: https://openrouter.ai/api/v1/chat/completions
[ERROR] API call failed: 404 Client Error: Not Found for url: https://openrouter.ai/api/v1/chat/completions
[ERROR] API call failed: 404 Client Error: Not Found for url: https://openrouter.ai/api/v1/chat/completions
[ERROR] API call failed: 404 Client Error: Not Found for url: https://openrouter.ai/api/v1/chat/completions

  Results:
    Score: 0.500
    Metrics: {'answer_relevance': 0.5, 'groundedness': 0.5, 'plan_quality': 0.5}
    Answer: {"error": "404 Client Error: Not Found for url: https://openrouter.ai/api/v1/chat/completions"}...


In [15]:
# Step 3: Export and view trace
print("\n3. Export OTLP Trace:")
print("-" * 40)

otlp = instrumented.session.flush_otlp()
spans = otlp["resourceSpans"][0]["scopeSpans"][0]["spans"]

print(f"  Total spans: {len(spans)}")
print(f"\n  Span Summary:")
for i, span in enumerate(spans):
    attrs = {a["key"]: a["value"]["stringValue"] for a in span.get("attributes", [])}
    is_ignored = "trace.temporal_ignore" in attrs
    marker = "[CHILD]" if is_ignored else "[NODE]"
    print(f"    {i+1}. {marker} {span['name']}")

# Save single-execution trace to RUN_FOLDER
single_trace_path = os.path.join(RUN_FOLDER, "single_execution_trace.json")
with open(single_trace_path, "w") as f:
    json.dump(otlp, f, indent=2)
print(f"\n  Trace saved to: {single_trace_path}")


3. Export OTLP Trace:
----------------------------------------
  Total spans: 10

  Span Summary:
    1. [NODE] planner
    2. [CHILD] openrouter.chat.completion
    3. [NODE] researcher
    4. [CHILD] openrouter.chat.completion
    5. [NODE] synthesizer
    6. [CHILD] openrouter.chat.completion
    7. [NODE] evaluator
    8. [CHILD] openrouter.chat.completion
    9. [NODE] evaluation_metrics
    10. [CHILD] agentlightning.annotation


## 7. Demo: Optimization Loop

Demonstrate the optimization loop with multiple queries and iterations.

In [16]:
# =============================================================================
# DEMO: OPTIMIZATION LOOP
# =============================================================================

print("="*60)
print("DEMO: Optimization Loop")
print("="*60)

# Create fresh instrumented graph
instrumented = instrument_graph(
    service_name="optimization-demo",
    trainable_keys={"planner", "synthesizer"},
)

# Run optimization (ONE function call!)
queries = [
    "What is artificial intelligence?",
    "Explain quantum computing basics.",
]

opt_result = optimize_langgraph(
    instrumented,
    queries=queries,
    iterations=2,
)

print("\n" + "="*60)
print("OPTIMIZATION RESULTS")
print("="*60)
print(f"  Baseline Score: {opt_result.baseline_score:.3f}")
print(f"  Best Score: {opt_result.best_score:.3f}")
print(f"  Best Iteration: {opt_result.best_iteration}")
print(f"  Score History: {[f'{s:.3f}' for s in opt_result.score_history]}")

DEMO: Optimization Loop
  Running baseline...
    Query 1/2: What is artificial intelligence?...
[ERROR] API call failed: 404 Client Error: Not Found for url: https://openrouter.ai/api/v1/chat/completions
[ERROR] API call failed: 404 Client Error: Not Found for url: https://openrouter.ai/api/v1/chat/completions
[ERROR] API call failed: 404 Client Error: Not Found for url: https://openrouter.ai/api/v1/chat/completions
[ERROR] API call failed: 404 Client Error: Not Found for url: https://openrouter.ai/api/v1/chat/completions
      Score: 0.500
    Query 2/2: Explain quantum computing basics....
[ERROR] API call failed: 404 Client Error: Not Found for url: https://openrouter.ai/api/v1/chat/completions
[ERROR] API call failed: 404 Client Error: Not Found for url: https://openrouter.ai/api/v1/chat/completions
[ERROR] API call failed: 404 Client Error: Not Found for url: https://openrouter.ai/api/v1/chat/completions
[ERROR] API call failed: 404 Client Error: Not Found for url: https://openro

## 8. View Trace Output

View the detailed OTLP trace output.

In [17]:
# =============================================================================
# VIEW DETAILED TRACE
# =============================================================================

print("="*60)
print("DETAILED TRACE OUTPUT")
print("="*60)

# Get trace from last optimization run
if opt_result.all_runs and opt_result.all_runs[0]:
    sample_otlp = opt_result.all_runs[0][0].otlp  # Baseline, first query
    spans = sample_otlp["resourceSpans"][0]["scopeSpans"][0]["spans"]
    
    print(f"\nSample trace (baseline, query 1):")
    print(f"Total spans: {len(spans)}")
    
    for i, span in enumerate(spans[:6]):  # Show first 6 spans
        print(f"\n--- Span {i+1}: {span['name']} ---")
        attrs = {a["key"]: a["value"]["stringValue"] for a in span.get("attributes", [])}
        for key, value in list(attrs.items())[:5]:  # Show first 5 attributes
            display_value = value[:80] + "..." if len(value) > 80 else value
            print(f"  {key}: {display_value}")

DETAILED TRACE OUTPUT

Sample trace (baseline, query 1):
Total spans: 10

--- Span 1: planner ---
  param.planner_prompt: You are a planning agent. Given a user query, create a simple plan.
Output a JSO...
  param.planner_prompt.trainable: True
  gen_ai.model: meta-llama/llama-3.1-8b-instruct:free
  inputs.gen_ai.prompt: You are a planning agent. Given a user query, create a simple plan.
Output a JSO...

--- Span 2: openrouter.chat.completion ---
  trace.temporal_ignore: true
  gen_ai.operation.name: chat
  gen_ai.provider.name: openrouter
  gen_ai.output.preview: {"error": "404 Client Error: Not Found for url: https://openrouter.ai/api/v1/cha...

--- Span 3: researcher ---
  gen_ai.model: meta-llama/llama-3.1-8b-instruct:free
  inputs.gen_ai.prompt: Provide key facts about: What is artificial intelligence?

--- Span 4: openrouter.chat.completion ---
  trace.temporal_ignore: true
  gen_ai.operation.name: chat
  gen_ai.provider.name: openrouter
  gen_ai.output.preview: {"error": "404 Cl

In [18]:
# =============================================================================
# SAVE TRACES TO GOOGLE DRIVE (or local fallback)
# =============================================================================
# Uses the RUN_FOLDER created earlier (Google Drive on Colab, local otherwise).
# =============================================================================

print("="*60)
print("SAVE TRACES TO FILES")
print("="*60)

if opt_result.all_runs and opt_result.all_runs[0]:
    # --- sample trace ---
    sample_otlp = opt_result.all_runs[0][0].otlp
    trace_path = os.path.join(RUN_FOLDER, "notebook_trace_output.json")
    with open(trace_path, "w") as f:
        json.dump(sample_otlp, f, indent=2)
    print(f"  Saved: {trace_path}")

    # --- all optimization traces ---
    all_traces = []
    for iter_idx, runs in enumerate(opt_result.all_runs):
        iter_name = "baseline" if iter_idx == 0 else f"iteration_{iter_idx}"
        for run_idx, run in enumerate(runs):
            all_traces.append({
                "iteration": iter_name,
                "query_index": run_idx,
                "score": run.score,
                "otlp": run.otlp,
            })

    opt_path = os.path.join(RUN_FOLDER, "notebook_optimization_traces.json")
    with open(opt_path, "w") as f:
        json.dump(all_traces, f, indent=2)
    print(f"  Saved: {opt_path}")
    print(f"  Total traces saved: {len(all_traces)}")

    # --- summary ---
    summary = {
        "baseline_score": opt_result.baseline_score,
        "best_score": opt_result.best_score,
        "best_iteration": opt_result.best_iteration,
        "score_history": opt_result.score_history,
    }
    summary_path = os.path.join(RUN_FOLDER, "run_summary.json")
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)
    print(f"  Saved: {summary_path}")

print(f"\n  All results persisted in: {RUN_FOLDER}")

SAVE TRACES TO FILES
  Saved: notebook_trace_output.json
  Saved: notebook_optimization_traces.json
  Total traces saved: 6


## Summary

This notebook demonstrated:

1. **`instrument_graph()`** - ONE function call to add OTEL instrumentation to any LangGraph
2. **`optimize_langgraph()`** - ONE function call for running optimization loops
3. **Dual Semantic Conventions** - Spans compatible with both Trace TGJ and Agent Lightning
4. **OTLP Export** - Full trace export to JSON format

### Before vs After

| Aspect | Before | After |
|--------|--------|-------|
| Instrumentation | ~200 lines boilerplate | 1 function call |
| Optimization loop | ~150 lines | 1 function call |
| Total boilerplate | ~645 lines | ~10 lines |

### Next Steps

- Set `USE_STUB_LLM = False` and add your OpenRouter API key for real LLM calls
- Examine the generated trace files for detailed span information
- Integrate with the Trace framework for optimization via TGJ